<a href="https://colab.research.google.com/github/tomonari-masada/course2026-nlp/blob/main/03_BoW_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BoWによるテキスト分類
* BoWでも良い性能を出せることが多い。
  * LLMを使って文書分類するときは、BoW+SVMの性能と比較した方が良い。
  * なぜなら、分類性能に大きな差がつかないことも、しばしばあるので。

## 準備

### spaCyの最小限の機能のインストール

* 下記のようにすると、英語だけ扱えるようになる。

In [ ]:
#!pip install -U spacy
#!python -m spacy download en_core_web_sm

### spaCyの日本語形態素解析器のインストール


* 下記のようにすると、Sudachiという形態素解析器が使えるようになる。

In [ ]:
!python -m spacy download ja_core_news_sm

### 乱数のシードの設定
* transformersのset_seedを使うと便利。

In [ ]:
from transformers import set_seed
set_seed(42)

## データセットの取得

* ライブドアニュースコーパスのタイトルを使う。

In [ ]:
from datasets import load_dataset

ds = load_dataset("mteb/LivedoorNewsClustering")

In [ ]:
ds

In [ ]:
ds["train"][0]

In [ ]:
# labelsが0のsentencesを3つ示す
import random

label_id = 8

filtered_ds = ds["train"].filter(lambda x: x["labels"] == label_id)
random_indices = random.sample(range(len(filtered_ds)), 3)
for i in random_indices:
  print(filtered_ds[i]["sentences"])

* 9種類のクラスラベルを参考までに示しておく。

In [ ]:
category_names = [
    'dokujo-tsushin',
    'it-life-hack',
    'kaden-channel',
    'livedoor-homme',
    'movie-enter',
    'peachy',
    'smax',
    'sports-watch',
    'topic-news',
]

## 形態素解析

In [ ]:
ds["train"][1000]

* 一つのテキストでSudachiによる形態素解析を試してみる。

In [ ]:
import spacy

nlp = spacy.load("ja_core_news_sm")
doc = nlp(ds["train"][1000]["sentences"])
text = " ".join([token.lemma_ for token in doc])
print(text)

* トークンのなかに改行記号が混ざっている。

In [ ]:
text

* 訓練データ全体を形態素解析する。
  * TfidfVectorizerの入力として使うので・・・
  * 形態素を半角空白文字でつないで、一つの長い文字列にしておく。

In [ ]:
from tqdm.auto import tqdm

new_line_chars = ["\n", "\r", "\u2028", "\u2029", "\u0085"]

corpus_train = []
for text in tqdm(ds["train"]["sentences"]):
  doc = nlp(text)
  text = " ".join([token.lemma_ for token in doc])
  for new_line_char in new_line_chars:
    text = " ".join([line for line in text.split(new_line_char)])
  corpus_train.append(text)

* ファイルへ書き込む。

In [ ]:
with open('livedoor_train.txt', 'w') as f:
  f.write("\n".join(corpus_train) + "\n")

* ファイルから読み込む。

In [ ]:
with open('livedoor_train.txt', 'r') as f:
  corpus_train = f.read().splitlines()

In [ ]:
corpus_train[0]

* 検証データとテストデータも形態素解析する。

In [ ]:
corpus_val = []
for text in tqdm(ds["validation"]["sentences"]):
  doc = nlp(text)
  text = " ".join([token.lemma_ for token in doc])
  for new_line_char in new_line_chars:
    text = " ".join([line for line in text.split(new_line_char)])
  corpus_val.append(text)

In [ ]:
with open('livedoor_validation.txt', 'w') as f:
  f.write("\n".join(corpus_val) + "\n")

In [ ]:
with open('livedoor_validation.txt', 'r') as f:
  corpus_val = f.read().splitlines()

In [ ]:
corpus_test = []
for text in tqdm(ds["test"]["sentences"]):
  doc = nlp(text)
  text = " ".join([token.lemma_ for token in doc])
  for new_line_char in new_line_chars:
    text = " ".join([line for line in text.split(new_line_char)])
  corpus_test.append(text)

In [ ]:
with open('livedoor_test.txt', 'w') as f:
  f.write("\n".join(corpus_test) + "\n")

In [ ]:
with open('livedoor_test.txt', 'r') as f:
  corpus_test = f.read().splitlines()

## TF-IDFベクトルの計算

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(min_df=10, max_df=0.2)
X_train = vectorizer.fit_transform(corpus_train)

In [ ]:
X_train.shape

# 課題
* ライブドアニュースコーパスを分類する分類器を作ろう。
* ベクトル化にはTfidfVectorizerを使うこと。
  * 言語モデルは次回使いますので。